<a href="https://colab.research.google.com/github/alreguer/Entorno-de-pruebas/blob/main/practica2y3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Clasificación, resumen y extracción de la información**

# **Prácticas 2 y 3**

En las sesiones de prácticas segunda y tercera vamos a programar utilidades básicas para la recuperación de la información y los primeros mecanismos de vectorización de textos. En particular, programaremos índices inversos y las métricas tf-idf.


En este cuaderno Colab, haremos uso de programas que hayáis diseñado en la Práctica 1. La forma más directa de hacer este reuso es que copiéis el código del otro cuaderno y lo peguéis en este.

Para los ejemplos, vamos a trabajar con la igualdad literal de cadenas:

In [2]:
def coincide (cadena1, cadena2):
  return(cadena1 == cadena2)

Dejaremos para los ejercicios opcionales que exploréis las posibilidades que ofrecen otras definiciones de "coincide" más sofisticadas (con *lematización*, o sinonimias, etc.)

Conviene mantener cargada la posibilidad de *tokenizar*:

In [3]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Definimos el siguiente texto (extraído de un diario digital, sin ningún criterio especial) para ir utilizándolo en los ejemplos:

In [4]:
texto = "El número de turistas que han visitado España en julio de este año han superado en un 3,1% las cifras pre pandemia. Con su llegada aumentan también los precios del alquiler y la masificación de entornos urbanos y naturales."

In [ ]:
word_tokenize(texto)

['El',
 'número',
 'de',
 'turistas',
 'que',
 'han',
 'visitado',
 'España',
 'en',
 'julio',
 'de',
 'este',
 'año',
 'han',
 'superado',
 'en',
 'un',
 '3,1',
 '%',
 'las',
 'cifras',
 'pre',
 'pandemia',
 '.',
 'Con',
 'su',
 'llegada',
 'aumentan',
 'también',
 'los',
 'precios',
 'del',
 'alquiler',
 'y',
 'la',
 'masificación',
 'de',
 'entornos',
 'urbanos',
 'y',
 'naturales',
 '.']

Nuestro objetivo ahora es, dada una colección textos/documentos, organizados como una lista de cadenas
(y utilizando como identificador su posición en la lista, comenzando en cero), recuperar en qué documentos aparece un término. Para ello, necesitaremos la función "apareceEnTexto" de la práctica anterior (y todas en las que esta se apoye, claro).

Lo hacemos primero sin utilizar ninguna estructura de datos auxiliar.

In [5]:
def aparece(cadena, listaCadenas):
  for item in listaCadenas:
    if coincide(cadena, item):
      return True
  return False

In [6]:
def apareceEnTexto(cadena,texto):
  return aparece(cadena, word_tokenize(texto))

**Ejercicio 1**. Programar una función que, dada una cadena y una colección de textos estructurada como una lista de cadenas, devuelva una lista con los (índices en la lista de entrada de los) textos en los que aparece la cadena que es el primer argumento.

In [7]:
def localizaEnTextos(cadena, textos):
    indices = []
    for i in range(len(textos)):
        if apareceEnTexto(cadena, textos[i]):
            indices.append(i)
    return indices

In [ ]:
localizaEnTextos('julio',[texto])

[0]

In [ ]:
localizaEnTextos('mayo',[texto])

[]

In [ ]:
localizaEnTextos('El',[texto])

[0]

Definimos ahora otros textos, para poder constituir con ellos una lista de textos (un repositorio o colección o corpus o *dataset*).

In [8]:
texto1 = "El calentamiento global multiplica por cuatro la intensidad de las lluvias torrenciales en España"

In [9]:
texto2 = "Ciudadano Musk: el más rico del mundo culmina la transformación de Twitter en su aparato de influencia global"

In [10]:
texto3 = "La justicia francesa avala la retirada de ayudas públicas a un medio por difundir bulos sobre la salud"

In [11]:
texto4 = "Estafas, publicidad y miedo a la interacción: por qué cada vez se contesta menos a números que no conocemos"

In [12]:
textos=[texto,texto1,texto2,texto3,texto4]

In [ ]:
textos

['El número de turistas que han visitado España en julio de este año han superado en un 3,1% las cifras pre pandemia. Con su llegada aumentan también los precios del alquiler y la masificación de entornos urbanos y naturales.',
 'El calentamiento global multiplica por cuatro la intensidad de las lluvias torrenciales en España',
 'Ciudadano Musk: el más rico del mundo culmina la transformación de Twitter en su aparato de influencia global',
 'La justicia francesa avala la retirada de ayudas públicas a un medio por difundir bulos sobre la salud',
 'Estafas, publicidad y miedo a la interacción: por qué cada vez se contesta menos a números que no conocemos']

In [ ]:
localizaEnTextos('la',textos)

[0, 1, 2, 3, 4]

In [ ]:
localizaEnTextos('de',textos)

[0, 1, 2, 3]

In [ ]:
localizaEnTextos('no',textos)

[4]

In [ ]:
localizaEnTextos('en',textos)

[0, 1, 2]

El problema de hacerlo así es que cada vez que preguntemos por un término tenemos que recorrer toda la colección de textos. Una forma más eficaz es calcular un *índice inverso*: un diccionario que, dado un término, indica los documentos/textos en los que ese término aparece. Ese diccionario/tabla se denomina "índice inverso" y aunque es costoso en tiempo construirlo, luego la consulta es muy eficaz.

Para comenzar a construir el índice inverso, fijamos un vocabulario de partida, que podría ser calculado (esto puede ser gestionado de modo dinámico: cuando llegue una consulta por un término "nuevo" se podría añadir al diccionario, de forma que el vocabulario estará constituido por las claves del diccionario).

Para tener un elemento de comparación cuando vayamos obteniendo resultados, vamos a fijar el siguiente vocabulario (fijaos en la presencia de *stop words*, que en la mayoría de las aplicaciones prácticas no serían parte de ningún vocabulario).

In [13]:
vocabulario = ["la", "no", "en", "de", "alquiler","salud", "calentamiento", "mundo"]

**Ejercicio 2**. Programar una función que, dado un vocabulario y una colección de textos estructurada como una lista de cadenas, devuelva un diccionario con claves las palabras del vocabulario y con valor la lista de índices de los textos en los que aparece esa palabra (utilizad "localizaEnTextos").

In [14]:
def indiceInverso(vocabulario,textos):
  diccionario = dict()
  for palabra in vocabulario:
    diccionario[palabra] = localizaEnTextos(palabra, textos)
  return diccionario


In [15]:
ii = indiceInverso(vocabulario,textos)

In [ ]:
ii

{'la': [0, 1, 2, 3, 4],
 'no': [4],
 'en': [0, 1, 2],
 'de': [0, 1, 2, 3],
 'alquiler': [0],
 'salud': [3],
 'calentamiento': [1],
 'mundo': [2]}

Una vez que tenemos construido un índice inverso, la extracción de información (es decir, determinar en qué textos aparece un término) es directa y muy sencilla.

**Ejercicio 3**. Programar una función que, dada una cadena y un índice inverso (un diccionario), devuelva una lista con los índices de los textos en los que aparece la cadena que es el primer argumento.

In [16]:
def localizaEnIndiceInverso(cadena,ii):
  lista = []
  for i in ii[cadena]:
    lista.append(i)
  return lista

In [17]:
localizaEnIndiceInverso("calentamiento",ii)

[1]

In [ ]:
localizaEnIndiceInverso("de",ii)

[0, 1, 2, 3]

In [ ]:
localizaEnIndiceInverso("la",ii)

[0, 1, 2, 3, 4]

Descomentad la siguiente celda y ejecutadla.

In [ ]:
localizaEnIndiceInverso("mayo",ii) # ¿error?

KeyError: 'mayo'

**Ejercicio opcional 1**. Si no habéis tenido precaución, al invocar la función anterior con una cadena que no esté en el vocabulario (o que no esté entre las claves del índice inverso; ¿por qué podría no ser lo mismo?), situación perfectamente razonable, por otra parte, se producirá un error. Aquí se pide programarla para evitar ese error de tres modos diferentes (uno de ellos, o algún otro alternativo que se os ocurra, deberá ser incorporado a vuestra definición, pues no es admisible programar una función que produzca un error ante entradas plausibles).

1. Utilizando el atributo .keys() del diccionario.

2. Utilizando la forma de acceso .get al diccionario.

3. Utilizando excepciones (este es el modo menos elegante, pero las excepciones son un recurso muy útil de los lenguajes de programación que conviene conocer).

In [18]:
def localizaEnIndiceInverso(cadena, ii):
    if cadena in ii.keys():
        return ii[cadena]
    else:
        return []

In [ ]:
localizaEnIndiceInverso("calentamiento",ii)

[1]

In [ ]:
localizaEnIndiceInverso("mayo",ii) # Ya no da error

[]

In [ ]:
localizaEnIndiceInverso("estupendísimo",ii) # Ya no da error

[]

**Ejercicio opcional 2 (abierto)**. Diseñar cómo podría programarse un sistema de búsquedas booleanas a partir de un índice inverso.

Como vocabulario: 'cosas' e 'ideas'.

Como repositorio/corpus/dataset:

textos[0]='cosas veredes'

textos[1]='los hechos son ideas y las ideas son cosas'

textos[2]='ideas son amores'

Ejemplos de posibles ejecuciones:

selecciona("(not 'ideas')",ii)

—> [1]

selecciona("('ideas' and 'cosas')",ii)

—> [2]

selecciona("(('cosas' and (not 'ideas')) or ((not 'cosas') and 'ideas'))",ii)

—> [1,3]


PARTE 1:	Una función que pase del lenguaje externo (la cadena con paréntesis) al lenguaje interno (una lista de listas).

In [44]:
def externoAinterno(expresion):

    expresion = expresion.strip()

    # si no hay paréntesis, es una palabra
    if "(" not in expresion:
        return expresion

    # quitamos paréntesis exteriores
    expresion = expresion[1:-1].strip()

    # caso (not X)
    if expresion.startswith("not "):
        return ["not", externoAinterno(expresion[4:])]

    # buscar operador principal
    nivel = 0
    for i in range(len(expresion)):

        if expresion[i] == "(":
            nivel = nivel + 1

        elif expresion[i] == ")":
            nivel = nivel - 1

        elif nivel == 0:

            if expresion[i:i+5] == " and ":
                izquierda = expresion[:i]
                derecha = expresion[i+5:]
                return [externoAinterno(izquierda), "and", externoAinterno(derecha)]

            if expresion[i:i+4] == " or ":
                izquierda = expresion[:i]
                derecha = expresion[i+4:]
                return [externoAinterno(izquierda), "or", externoAinterno(derecha)]

    return None

In [45]:
externoAinterno("(ideas and cosas)")

['ideas', 'and', 'cosas']

In [38]:
externoAinterno("(not ideas)")

['not', 'ideas']

In [39]:
externoAinterno("(not 'ideas')")

['not', "'ideas'"]

In [46]:
externoAinterno("((cosas and (not ideas)) or ((not cosas) and ideas))")

[['cosas', 'and', ['not', 'ideas']], 'or', [['not', 'cosas'], 'and', 'ideas']]

PARTE 2:	Una función que diga si una expresión en el lenguaje interno es válida.

In [19]:
def expresionValida(expresion, indice_inverso):
  if type(expresion).__name__ == 'str':
    return expresion in indice_inverso

  elif type(expresion).__name__ == 'list':

    if len(expresion) == 2:
      return expresion[0] == "not" and expresionValida(expresion[1], indice_inverso)

    elif len(expresion) == 3:
      return (expresion[1] in ["and", "or"] and
              expresionValida(expresion[0], indice_inverso) and
              expresionValida(expresion[2], indice_inverso))
    else:
      return False

  else:
    return False

In [20]:
expresionValida("alquiler", ii)

True

In [ ]:
expresionValida("mayo", ii)

False

In [ ]:
expresionValida(["alquiler", "and", "calentamiento"], ii)

True

In [ ]:
expresionValida(["alquiler", "or", "calentamiento"], ii)

True

In [ ]:
expresionValida(["alquiler", "not", "calentamiento"], ii)

False

In [23]:
expresionValida(["alquiler", "and", "calentamiento", "mayo"], ii)

False

In [ ]:
expresionValida(["not", "calentamiento"], ii)

True

In [ ]:
expresionValida(["and", "calentamiento"], ii)

False

PARTE 3: Una función que, dada una expresión válida en el lenguaje interno (y dado un índice inverso), realiza el cálculo.


In [71]:
def calculo_ii(expresion, indice_inverso):
  if expresionValida(expresion, indice_inverso):

    resultados = []

    for cadena in expresion:
      resultados.append(localizaEnIndiceInverso(cadena, indice_inverso))

    return resultados

In [72]:
calculo_ii("calentamiento", ii)

[[], [], [], [], [], [], [], [], [], [], [], [], []]

Los índices inversos sirven para hacer búsquedas "exactas" ("booleanas", desde el punto de vista de la lógica); sin embargo en un contexto de *big data*, puede ser interesante recuperar solo los documentos más significativos, por medio de una prelación (*ranked retrieval*). Para abordar este problema, vamos a hacer una pequeña digresión sobre cómo representar los textos.

Si solo estamos interesados en un vocabulario fijado (que puede ser tan amplio como queramos), podemos quedarnos solamente, de cada texto/documento, con la información de qué palabras del vocabulario pertenecen a él. Esa información, aunque es mucho más pobre que el texto en sí, es suficiente, por ejemplo, para establecer el índice inverso. Esa representación de cada frase se denomina SoW (Set of Words).

**Ejercicio 4**. Programar una función que, dado un texto (una cadena de caracteres) y un vocabulario (una lista de cadenas, sin repeticiones), devuelva la lista de palabras del vocabulario que aparecen en el texto (utilizad "apareceEnTexto"). ¿Lo podríais hacer en una única línea usando compresión de listas?

In [ ]:
def sow (texto, vocabulario):
  return [cadena for cadena in vocabulario if apareceEnTexto(cadena, texto)]


In [ ]:
sow(texto,vocabulario)

['la', 'en', 'de', 'alquiler']

Cada sow admite una representación como conjunto (como en el ejemplo anterior) o bien como un vector de dimensión la cardinalidad del vocabulario (representación densa) o como un diccionario (representación dispersa; *sparse* en inglés).

**Ejercicio 5**. Programar una función que, dado un texto (una cadena de caracteres) y un vocabulario (una lista de cadenas, sin repeticiones), devuelva un vector de 0s y 1s, con un 1 en la posición en la que un término del vocabulario aparezca en el texto (utilizad "apareceEnTexto"). Aclaración en clase: representación densa. Poner un 1 si la palabra está (el 1 equivaldría a True). Y si la palabra no está, un 0.

In [ ]:
def sow_vector (texto, vocabulario):
  vector = []
  for cadena in vocabulario:
    if apareceEnTexto(cadena, texto):
      vector.append(1)
    else:
      vector.append(0)
  return vector


In [ ]:
sow_vector(texto,vocabulario)

[1, 0, 1, 1, 1, 0, 0, 0]

Se lee mejor en paralelo (intentad entender el código):

In [ ]:
def muestra_sow_vector(texto,vocabulario):
  print("La representación de: \n" + texto + "\nes la siguiente:")
  for cadena,valor in zip(vocabulario,sow_vector(texto, vocabulario)):
    print(cadena + " : " + str(valor))

In [ ]:
muestra_sow_vector(texto,vocabulario)

La representación de: 
El número de turistas que han visitado España en julio de este año han superado en un 3,1% las cifras pre pandemia. Con su llegada aumentan también los precios del alquiler y la masificación de entornos urbanos y naturales.
es la siguiente:
la : 1
no : 0
en : 1
de : 1
alquiler : 1
salud : 0
calentamiento : 0
mundo : 0


**Ejercicio 6**. Programar una función que, dado un texto (una cadena de caracteres) y un vocabulario (una lista de cadenas, sin repeticiones), devuelva un diccionario en el que las claves sean los términos del vocabulario que aparezcan en el texto (utilizad "apareceEnTexto"). Aclaración en clase: Hacerlo en representación dispersa/sparse (si la palabra no aparece, el resultado será un diccionario vacío). Claves: cualquier cosa.

In [ ]:
def sow_dic (texto, vocabulario):
  diccionario = dict()
  for cadena in vocabulario:
    if apareceEnTexto(cadena, texto):
      diccionario[cadena] = (1)
  return diccionario

In [ ]:
sow_dic(texto,vocabulario)

{'la': 1, 'en': 1, 'de': 1, 'alquiler': 1}

In [ ]:
sow_dic(texto1,vocabulario)

{'la': 1, 'en': 1, 'de': 1, 'calentamiento': 1}

In [ ]:
texto5 = "Carpe diem, me dijo"

In [ ]:
sow_dic(texto5,vocabulario)

{}

In [ ]:
sow_dic(texto,vocabulario).keys()

dict_keys(['la', 'en', 'de', 'alquiler'])

In [ ]:
vocabulario

['la', 'no', 'en', 'de', 'alquiler', 'salud', 'calentamiento', 'mundo']

La información contenida en los sows es suficiente para construir el índice inverso. En primer lugar, vamos detectar en qué sows (implementados como diccionarios) aparece un término.

**Ejercicio 7**. Programar una función que, dada una cadena y una lista de sows (como diccionarios), devuelva una lista con los índices de los sows en los que aparezca la cadena de entrada.


In [ ]:
def localizaEnSows(cadena,sows):
  indices = []
  for i in range(len(sows)):
    if cadena in sows[i].keys():
      indices.append(i)
  return indices

In [ ]:
localizaEnSows('de',[sow_dic(texto,vocabulario) for texto in textos])

[0, 1, 2, 3]

In [ ]:
localizaEnSows('mundo',[sow_dic(texto,vocabulario) for texto in textos])

[2]

In [ ]:
localizaEnSows('justicia',[sow_dic(texto,vocabulario) for texto in textos])

[]

**Ejercicio 8**. Programar una función que, dado un vocabulario y una lista de sows (como diccionarios), devuelva el índice inverso de la colección de textos (representada por la lista de sows) del vocabulario (utilizando "localizaEnSows").

In [ ]:
def iiFromSows(vocabulario,sows):
  diccionario = dict()
  for cadena in vocabulario:
    diccionario[cadena] = localizaEnSows(cadena,sows)
  return diccionario

In [ ]:
iiFromSows(vocabulario,[sow_dic(texto,vocabulario) for texto in textos])

{'la': [0, 1, 2, 3, 4],
 'no': [4],
 'en': [0, 1, 2],
 'de': [0, 1, 2, 3],
 'alquiler': [0],
 'salud': [3],
 'calentamiento': [1],
 'mundo': [2]}

Al pasar de un texto a un sow hemos perdido mucha información (aunque hemos mantenido la suficiente como para construir el índice inverso) de tipo gramatical, pero también otra que puede ser interesante si queremos hacer una extracción de la información con prelación: el número de veces que una palabra aparece en un texto. Almacenar esa información es lo que se conoce como pasar a un modo de representación BoW: *Bag of Words*. La representación puede ser tanto densa como dispersa. Programamos solo al segunda.

**Ejercicio 9**. Programar una función que, dado un texto y un vocabulario, devuelva un diccionario que represente el bow del texto respecto al vocabulario (utilícese la función "frecuencia" de la Práctica 1).

In [ ]:
def frecuencia(cadena, listaCadenas):
    contador = 0
    for item in listaCadenas:
        if coincide(cadena, item):
            contador = contador + 1
    return contador

In [ ]:
def bow_dic (texto, vocabulario):
  diccionario = dict()
  for cadena in vocabulario:
    if apareceEnTexto(cadena, texto):
      diccionario[cadena] = frecuencia(cadena, word_tokenize(texto))
  return diccionario


In [ ]:
def bow_dic_denso (texto, vocabulario):
  diccionario = dict()
  for cadena in vocabulario:
    diccionario[cadena] = frecuencia(cadena, word_tokenize(texto))
  return diccionario


In [ ]:
bow_dic(texto,vocabulario)

{'la': 1, 'en': 2, 'de': 3, 'alquiler': 1}

In [ ]:
bow_dic_denso(texto,vocabulario)

{'la': 1,
 'no': 0,
 'en': 2,
 'de': 3,
 'alquiler': 1,
 'salud': 0,
 'calentamiento': 0,
 'mundo': 0}

Ahora la idea es, dado un texto como una lista de bows, organizar el índice inverso ordenando los documentos en los que aparece un términos de mayor a menor frecuencia según indica cada bow.

Sin embargo, esto no tiene en cuenta la importancia del término dentro de toda la colección de documentos. Es decir, queremos dar más peso a aquellos términos dentro de un documento que sean frecuentes en él, pero no aparezcan demasiado en el resto de documentos. Para eso se introduce la medida tf-idf de cada término en cada documento de una colección de documentos.

**Ejercicio 10**. Programar una función que, dada una cadena y un lista de bows, devuelva el número de documentos de la lista en la que la cadena aparece.

In [ ]:
def df (cadena,bows):  # document frequency
  contador = 0
  for bow in bows:
    if bow.get(cadena, 0) > 0:
      contador = contador + 1
  return contador

In [ ]:
bows = [bow_dic(texto,vocabulario) for texto in textos]

In [ ]:
bows

[{'la': 1, 'en': 2, 'de': 3, 'alquiler': 1},
 {'la': 1, 'en': 1, 'de': 1, 'calentamiento': 1},
 {'la': 1, 'en': 1, 'de': 2, 'mundo': 1},
 {'la': 2, 'de': 1, 'salud': 1},
 {'la': 1, 'no': 1}]

In [ ]:
df("la",bows)

5

In [ ]:
df("en",bows)

3

In [ ]:
df("salud",bows)

1

In [ ]:
df("mayo",bows)

0

**Ejercicio opcional 3**. Vamos a calcular la frecuencia de un término dentro de una colección (*collection frequency*, cf). De dos formas:

1. Utilizando la representación dispersa de bows, por medio de diccionario, como hemos venido haciendo.

2. Utilizando una representación densa de bows (como vectores de números) y el tratamiento de matrices que planteamos en el Ejercicio opcional 2 de la Práctica 1.

PRIMERA MANERA (accediendo a las claves del diccionario):

In [ ]:
def cf(cadena,bows): # collection frequency (a partir de bows dispersos)
  contador = 0
  for bow in bows:
    contador = contador + bow.get(cadena, 0)
  return contador

In [ ]:
cf("la",bows)

6

In [ ]:
cf("de",bows)

7

In [ ]:
cf("alquiler",bows)

1

In [ ]:
cf("mayo",bows)

0

SEGUNDA MANERA (utilizando una representación densa de bows):

In [ ]:
def bow_vector(texto, vocabulario):
  vector = []
  for cadena in vocabulario:
    vector.append(frecuencia(cadena, word_tokenize(texto)))
  return vector

In [ ]:
vocabulario

['la', 'no', 'en', 'de', 'alquiler', 'salud', 'calentamiento', 'mundo']

In [ ]:
texto

'El número de turistas que han visitado España en julio de este año han superado en un 3,1% las cifras pre pandemia. Con su llegada aumentan también los precios del alquiler y la masificación de entornos urbanos y naturales.'

In [ ]:
bow_vector(texto,vocabulario)

[1, 0, 2, 3, 1, 0, 0, 0]

In [ ]:
def cf_denso (cadena, textos, vocabulario):
  matriz = [] # Construimos la matriz de vectores de bows
  for texto in textos:
    matriz.append(bow_vector(texto, vocabulario))

  num_filas = len(matriz)
  num_columnas = len(matriz[0])

  contador = 0

  for i in range(num_columnas): # Recorremos columnas
    if coincide(vocabulario[i], cadena):
      for j in range(num_filas): # Recorremos filas
        contador = contador + matriz[j][i]
  return contador

In [ ]:
cf_denso("la",textos,vocabulario)

6

In [ ]:
cf_denso("de",textos,vocabulario)

7

In [ ]:
cf_denso("alquiler",textos,vocabulario)

1

In [ ]:
cf_denso("mayo",textos,vocabulario)

0

Para definir el valor idf vamos a tener que calcular logaritmos. Necesitamos importar la librería math:

In [ ]:
import math

El logaritmo se utiliza para *suavizar* el crecimiento (o decrecimiento) brusco de los números. Cuando solo nos interesa comparar dos números (para saber cuál de ellos es más grande) podemos comparar sus logaritmos, sirviendo para el mismo fin (porque el logaritmo es una función creciente: x < y <==> log(x) < log(y)), y limitando el peligro de "overflow" o "underflow".

In [ ]:
x = 10
for i in range(10):
  print ('Valor: ' + str(x) + "  | su logaritmo: " + str(math.log(x)))
  print ('Valor: ' + str(1/x) + "  | su logaritmo: " + str(math.log(1/x)))
  x = x * 100


Valor: 10  | su logaritmo: 2.302585092994046
Valor: 0.1  | su logaritmo: -2.3025850929940455
Valor: 1000  | su logaritmo: 6.907755278982137
Valor: 0.001  | su logaritmo: -6.907755278982137
Valor: 100000  | su logaritmo: 11.512925464970229
Valor: 1e-05  | su logaritmo: -11.512925464970229
Valor: 10000000  | su logaritmo: 16.11809565095832
Valor: 1e-07  | su logaritmo: -16.11809565095832
Valor: 1000000000  | su logaritmo: 20.72326583694641
Valor: 1e-09  | su logaritmo: -20.72326583694641
Valor: 100000000000  | su logaritmo: 25.328436022934504
Valor: 1e-11  | su logaritmo: -25.328436022934504
Valor: 10000000000000  | su logaritmo: 29.933606208922594
Valor: 1e-13  | su logaritmo: -29.933606208922594
Valor: 1000000000000000  | su logaritmo: 34.538776394910684
Valor: 1e-15  | su logaritmo: -34.538776394910684
Valor: 100000000000000000  | su logaritmo: 39.14394658089878
Valor: 1e-17  | su logaritmo: -39.14394658089878
Valor: 10000000000000000000  | su logaritmo: 43.74911676688687
Valor: 1e-19

**Ejercicio 11**. Programar una función que, dada una cadena y un lista de bows, devuelva el valor idf de la cadena (respecto a alguna de las definiciones explicadas en teoría).

In [ ]:
#def idf (cadena, bows): # No utilizar, colapsa a cero
  n = len(bows)
  df_valor = df(cadena, bows)

  return math.log(n / df_valor)

In [ ]:
 n = len(bows)
 n

5

In [ ]:
idf("la",bows)

0.0

In [ ]:
idf("en",bows)

0.5108256237659907

In [ ]:
idf("salud",bows)

1.6094379124341003

Descomentad y ejecutad la siguiente celda.

In [ ]:
idf("mayo",bows) # ¿error?

0

DOS POSIBLES SOLUCIONES PARA EVITAR LA DIVISIÓN POR CERO:

In [ ]:
def idf (cadena, bows):
  n = len(bows)
  df_valor = df(cadena, bows)

  if df_valor == 0:
    return 0

  return math.log(n / df_valor)

In [ ]:
def idf (cadena, bows):
  n = len(bows)
  df_valor = df(cadena, bows)

  return math.log((n + 1) / (df_valor + 1))

In [ ]:
def idf (cadena, bows):
  n = len(bows)
  df_valor = df(cadena, bows)

  return math.log(n / (df_valor + 1))

La representación con el modelo BoW ya calcula la frecuencia del término en un texto (tf, *term frequency*). Esencialmente, hay que devolver bow[cadena]. Pero hay que tener cuidado de que si la cadena por la que se pregunta no aparece en el texto, se devuelva 0:

In [ ]:
def tf (cadena, bow):
  return (bow.get(cadena,0))

In [ ]:
tf("de",bows[0])

3

In [ ]:
tf("en",bows[0])

2

In [ ]:
tf("mayo",bows[0])

0

Ahora para calcular el valor de tf-idf basta multiplicar tf por idf.

**Ejercicio 12**. Programar una función que, dada una cadena, un bow y un lista de bows, devuelva el valor tf-idf de la cadena en el texto bow respecto a la colección de textos bows (una única línea).

In [ ]:
def tfxidf (cadena,bow,bows):
  return tf(cadena, bow) * idf(cadena, bows)

In [ ]:
bows

[{'la': 1, 'en': 2, 'de': 3, 'alquiler': 1},
 {'la': 1, 'en': 1, 'de': 1, 'calentamiento': 1},
 {'la': 1, 'en': 1, 'de': 2, 'mundo': 1},
 {'la': 2, 'de': 1, 'salud': 1},
 {'la': 1, 'no': 1}]

In [ ]:
tfxidf("la",bows[0],bows)

0.0

In [ ]:
tfxidf("de",bows[0],bows)

0.5469646703818638

In [ ]:
tfxidf("en",bows[0],bows)

0.8109302162163288

In [ ]:
tfxidf("salud",bows[0],bows)

0.0

In [ ]:
tfxidf("salud",bows[3],bows)

1.0986122886681098

In [ ]:
tfxidf("calentamiento",bows[0],bows)

0.0

In [ ]:
tfxidf("calentamiento",bows[1],bows)

1.0986122886681098

In [ ]:
tfxidf("mayo",bows[2],bows)

0.0

In [ ]:
tfxidf("mundo",bows[2],bows)

1.0986122886681098

**Ejercicio 13**. Programar una función que, dado una bow y un lista de bows, devuelva un diccionario en el que a cada término del bow de entrada se le asocie su valor tf-idf.

In [ ]:
def tfxidf_dic(bow,bows):
  diccionario = dict()
  for cadena in bow.keys():
    diccionario[cadena] = tfxidf(cadena, bow, bows)
  return diccionario

In [ ]:
len(bows)

5

In [ ]:
 tfxidf_dic(bows[0],bows)

{'la': 0.0,
 'en': 0.8109302162163288,
 'de': 0.5469646703818638,
 'alquiler': 1.0986122886681098}

In [ ]:
 tfxidf_dic(bows[1],bows)

{'la': 0.0,
 'en': 0.4054651081081644,
 'de': 0.1823215567939546,
 'calentamiento': 1.0986122886681098}

In [ ]:
 tfxidf_dic(bows[2],bows)

{'la': 0.0,
 'en': 0.4054651081081644,
 'de': 0.3646431135879092,
 'mundo': 1.0986122886681098}

In [ ]:
 tfxidf_dic(bows[3],bows)

{'la': 0.0, 'de': 0.1823215567939546, 'salud': 1.0986122886681098}

In [ ]:
 tfxidf_dic(bows[4],bows)

{'la': 0.0, 'no': 1.0986122886681098}

**Ejercicio 14**. Programar una función que, dada una lista de bows, devuelva una lista de diccionarios cada uno de los cuales corresponde al de valores tf-idf de cada uno de los bows de la lista de entrada.

In [ ]:
def tfxidf_model(bows):
  return [tfxidf_dic(bow,bows) for bow in bows]

In [ ]:
tfxidf_model(bows)

[{'la': 1.0986122886681098,
  'en': 2.643511679964639,
  'de': 3.6119184129778077,
  'alquiler': 1.6094379124341003},
 {'la': 1.0986122886681098,
  'en': 1.3217558399823195,
  'de': 1.203972804325936,
  'calentamiento': 1.6094379124341003},
 {'la': 1.0986122886681098,
  'en': 1.3217558399823195,
  'de': 2.407945608651872,
  'mundo': 1.6094379124341003},
 {'la': 2.1972245773362196,
  'de': 1.203972804325936,
  'salud': 1.6094379124341003},
 {'la': 1.0986122886681098, 'no': 1.6094379124341003}]

In [ ]:
bows

[{'la': 1, 'en': 2, 'de': 3, 'alquiler': 1},
 {'la': 1, 'en': 1, 'de': 1, 'calentamiento': 1},
 {'la': 1, 'en': 1, 'de': 2, 'mundo': 1},
 {'la': 2, 'de': 1, 'salud': 1},
 {'la': 1, 'no': 1}]

En las siguientes celdas vamos a generar el modelo tf-idf que proporciona la librería sklearn, y a extraerlo de forma que lo podamos comparar con el que acabamos de calcular.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer(stop_words=None, vocabulary=vocabulario) # para comparar, el mismo vocabulario

In [ ]:
feature_names = vectorizer.get_feature_names_out()

In [ ]:
feature_names

array(['la', 'no', 'en', 'de', 'alquiler', 'salud', 'calentamiento',
       'mundo'], dtype=object)

In [ ]:
tf_idf = vectorizer.fit_transform(textos)

In [ ]:
print(tf_idf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 17 stored elements and shape (5, 8)>
  Coords	Values
  (0, 0)	0.19654575253083423
  (0, 2)	0.5524763946578989
  (0, 3)	0.6971408403404858
  (0, 4)	0.41247333154673
  (1, 0)	0.3375338265523302
  (1, 2)	0.4743920160255332
  (1, 3)	0.39907351927997176
  (1, 6)	0.7083526362438907
  (2, 0)	0.27765951098422603
  (2, 2)	0.3902407546227053
  (2, 3)	0.6565656505710366
  (2, 7)	0.5826996618170748
  (3, 0)	0.7797586601687984
  (3, 3)	0.30730849100478064
  (3, 5)	0.5454703688085403
  (4, 0)	0.430165282498796
  (4, 1)	0.9027501480103624


Lo que devuelve "vectorizer.fit_transform" es una matriz dispersa (*sparse*):

In [ ]:
tf_idf[0,4]

np.float64(0.41247333154673)

Con la siguiente función pasamos una matriz a una lista de diccionarios:

In [ ]:
def sparseMatrixToDics (matrix, n, m, feature_names):
  res = [0] * n
  for i in range(0,n): # es igual que range(n)
    res[i]=dict()
    for j in range(0,m):
      valaux = matrix[i,j]
      if valaux != 0.0:
        res[i][feature_names[j]] = valaux
  return res

In [ ]:
sparseMatrixToDics (tf_idf, len(textos), len(vocabulario), feature_names)

[{'la': np.float64(0.19654575253083423),
  'en': np.float64(0.5524763946578989),
  'de': np.float64(0.6971408403404858),
  'alquiler': np.float64(0.41247333154673)},
 {'la': np.float64(0.3375338265523302),
  'en': np.float64(0.4743920160255332),
  'de': np.float64(0.39907351927997176),
  'calentamiento': np.float64(0.7083526362438907)},
 {'la': np.float64(0.27765951098422603),
  'en': np.float64(0.3902407546227053),
  'de': np.float64(0.6565656505710366),
  'mundo': np.float64(0.5826996618170748)},
 {'la': np.float64(0.7797586601687984),
  'de': np.float64(0.30730849100478064),
  'salud': np.float64(0.5454703688085403)},
 {'la': np.float64(0.430165282498796), 'no': np.float64(0.9027501480103624)}]

**Ejercicio opcional 4 (abierto)**. Comparar los valores obtenidos por el modelo tf-idf sklearn anterior con el que habéis programado antes. Si los valores no coinciden, intentad re-programar vuestro modelo empleando las formulas de sklearn en:

https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#

Lo consigáis o no, explicad a qué se pueden deber las discrepancias.

In [ ]:
def idf(cadena, bows):
    n = len(bows)
    df_valor = df(cadena, bows)

    # calculamos el df máximo de la colección
    max_df = 0
    for bow in bows:
        for termino in bow:
            df_actual = df(termino, bows)
            if df_actual > max_df:
                max_df = df_actual

    # normalizamos df
    df_norm = df_valor / max_df

    return math.log((n + 1) / (df_norm + 1))

In [ ]:
tfxidf_model(bows)

[{'la': 1.0986122886681098,
  'en': 2.643511679964639,
  'de': 3.6119184129778077,
  'alquiler': 1.6094379124341003},
 {'la': 1.0986122886681098,
  'en': 1.3217558399823195,
  'de': 1.203972804325936,
  'calentamiento': 1.6094379124341003},
 {'la': 1.0986122886681098,
  'en': 1.3217558399823195,
  'de': 2.407945608651872,
  'mundo': 1.6094379124341003},
 {'la': 2.1972245773362196,
  'de': 1.203972804325936,
  'salud': 1.6094379124341003},
 {'la': 1.0986122886681098, 'no': 1.6094379124341003}]

La librería de sklearn también permite vectorizar con bows. Aquí sí que se comprueba que nuestros cálculos coinciden con los de la librería.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
cvectorizer = CountVectorizer(vocabulary=vocabulario)

In [ ]:
skbows = cvectorizer.fit_transform(textos)

In [ ]:
print(skbows)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17 stored elements and shape (5, 8)>
  Coords	Values
  (0, 0)	1
  (0, 2)	2
  (0, 3)	3
  (0, 4)	1
  (1, 0)	1
  (1, 2)	1
  (1, 3)	1
  (1, 6)	1
  (2, 0)	1
  (2, 2)	1
  (2, 3)	2
  (2, 7)	1
  (3, 0)	3
  (3, 3)	1
  (3, 5)	1
  (4, 0)	1
  (4, 1)	1


In [ ]:
sparseMatrixToDics (skbows, len(textos), len(vocabulario), feature_names)

[{'la': np.int64(1),
  'en': np.int64(2),
  'de': np.int64(3),
  'alquiler': np.int64(1)},
 {'la': np.int64(1),
  'en': np.int64(1),
  'de': np.int64(1),
  'calentamiento': np.int64(1)},
 {'la': np.int64(1),
  'en': np.int64(1),
  'de': np.int64(2),
  'mundo': np.int64(1)},
 {'la': np.int64(3), 'de': np.int64(1), 'salud': np.int64(1)},
 {'la': np.int64(1), 'no': np.int64(1)}]

In [ ]:
bows

[{'la': 1, 'en': 2, 'de': 3, 'alquiler': 1},
 {'la': 1, 'en': 1, 'de': 1, 'calentamiento': 1},
 {'la': 1, 'en': 1, 'de': 2, 'mundo': 1},
 {'la': 2, 'de': 1, 'salud': 1},
 {'la': 1, 'no': 1}]

Terminamos esta sesión doble de prácticas con dos ejercicios opcionales.

**Ejercicio opcional 5**. Obtener el modelo tf-idf de una colección de textos extraídas de algún contexto real. Explicad qué vocabulario habéis elegido. Comparar con los resultados de sklearn. ¿Convendría cambiar la definición de "coincide"?

**Ejercicio opcional 6 (teórico)**. El algún momento de los programas que hemos ido escribiendo hemos perdido la flexibilidad que nos ofrece "coincide" (es decir, hemos comparado las cadenas con ==, la igualdad literal, y hemos perdido la posibilidad de comparar tras *stemming* o *lematización*, etc.). Detectad en qué funciones ha sido y proponed qué soluciones se os ocurren para poder recuperar esa flexibilidad.